In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Load the train and eval datasets
train_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/ml_benchmark/04_titanic/split_train.csv'
eval_data_path = 'D:\\LLM-Driven_AI-Studio\\MLAgent\\data\\benchmark/ml_benchmark/04_titanic/split_eval.csv'

train_df = pd.read_csv(train_data_path)
eval_df = pd.read_csv(eval_data_path)

# Display basic information about the datasets
print("Train Data Info:")
print(train_df.info())
print("\nEval Data Info:")
print(eval_df.info())

# Display the first few rows of the datasets
print("\nTrain Data Head:")
print(train_df.head())
print("\nEval Data Head:")
print(eval_df.head())

# Summary statistics for numerical columns
print("\nTrain Data Summary Statistics:")
print(train_df.describe())
print("\nEval Data Summary Statistics:")
print(eval_df.describe())

# Check for missing values
print("\nMissing Values in Train Data:")
print(train_df.isnull().sum())
print("\nMissing Values in Eval Data:")
print(eval_df.isnull().sum())

# Visualize the distribution of the target variable
plt.figure(figsize=(8, 6))
sns.countplot(x='Survived', data=train_df)
plt.title('Distribution of Survival in Train Data')
plt.show()

# Visualize the correlation matrix for numerical features
numerical_features = train_df.select_dtypes(include=[np.number]).columns
plt.figure(figsize=(12, 8))
sns.heatmap(train_df[numerical_features].corr(), annot=True, cmap='coolwarm')
plt.title('Correlation Matrix of Numerical Features in Train Data')
plt.show()

# Visualize the distribution of categorical features
categorical_features = train_df.select_dtypes(include=['object']).columns
for feature in categorical_features:
    plt.figure(figsize=(8, 6))
    sns.countplot(x=feature, data=train_df)
    plt.title(f'Distribution of {feature} in Train Data')
    plt.xticks(rotation=45)
    plt.show()


Train Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 712 entries, 0 to 711
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId  712 non-null    int64  
 1   Survived     712 non-null    int64  
 2   Pclass       712 non-null    int64  
 3   Name         712 non-null    object 
 4   Sex          712 non-null    object 
 5   Age          567 non-null    float64
 6   SibSp        712 non-null    int64  
 7   Parch        712 non-null    int64  
 8   Ticket       712 non-null    object 
 9   Fare         712 non-null    float64
 10  Cabin        174 non-null    object 
 11  Embarked     710 non-null    object 
dtypes: float64(2), int64(5), object(5)
memory usage: 66.9+ KB
None

Eval Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 179 entries, 0 to 178
Data columns (total 12 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   PassengerId

In [2]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df)
print("column_info")
print(column_info)


2025-09-10 03:37:08.074 | INFO     | metagpt.const:get_metagpt_package_root:29 - Package root set to D:\LLM-Driven_AI-Studio\MLAgent\experiments\DataInterpreter


column_info
{'Category': ['Name', 'Sex', 'Ticket', 'Cabin', 'Embarked'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Age', 'SibSp', 'Parch', 'Fare'], 'Datetime': [], 'Others': []}


In [3]:
from metagpt.tools.libs.data_preprocess import FillMissingValue, LabelEncode, StandardScale

# Copy the DataFrames to avoid modifying the original data
train_df_processed = train_df.copy()
eval_df_processed = eval_df.copy()

# Handle missing values
# Fill missing 'Age' with median
fill_age = FillMissingValue(features=['Age'], strategy='median')
train_df_processed = fill_age.fit_transform(train_df_processed)
eval_df_processed = fill_age.transform(eval_df_processed)

# Fill missing 'Cabin' with a placeholder
fill_cabin = FillMissingValue(features=['Cabin'], strategy='constant', fill_value='Unknown')
train_df_processed = fill_cabin.fit_transform(train_df_processed)
eval_df_processed = fill_cabin.transform(eval_df_processed)

# Fill missing 'Embarked' with the most frequent value
fill_embarked = FillMissingValue(features=['Embarked'], strategy='most_frequent')
train_df_processed = fill_embarked.fit_transform(train_df_processed)
eval_df_processed = fill_embarked.transform(eval_df_processed)

# Encode categorical variables
# Encode 'Sex' and 'Embarked' using label encoding
label_encode = LabelEncode(features=['Sex', 'Embarked'])
train_df_processed = label_encode.fit_transform(train_df_processed)
eval_df_processed = label_encode.transform(eval_df_processed)

# Normalize numerical features
# Standardize 'Age', 'SibSp', 'Parch', 'Fare'
standard_scale = StandardScale(features=['Age', 'SibSp', 'Parch', 'Fare'])
train_df_processed = standard_scale.fit_transform(train_df_processed)
eval_df_processed = standard_scale.transform(eval_df_processed)

# Display the processed data
print("Processed Train Data Head:")
print(train_df_processed.head())
print("\nProcessed Eval Data Head:")
print(eval_df_processed.head())


Processed Train Data Head:
   PassengerId  Survived  Pclass  ...      Fare    Cabin  Embarked
0          409         0       3  ... -0.485781  Unknown         2
1          481         0       3  ...  0.259653  Unknown         2
2          511         1       3  ... -0.486257  Unknown         1
3          610         1       1  ...  2.289948     C125         2
4          548         1       2  ... -0.369798  Unknown         0

[5 rows x 12 columns]

Processed Eval Data Head:
   PassengerId  Survived  Pclass  ...      Fare    Cabin  Embarked
0          206         0       3  ... -0.434577       G6         2
1           45         1       3  ... -0.483796  Unknown         1
2          822         1       3  ... -0.468872  Unknown         2
3          459         1       2  ... -0.433863  Unknown         2
4          796         0       2  ... -0.386231  Unknown         2

[5 rows x 12 columns]


In [4]:
from metagpt.tools.libs.data_preprocess import get_column_info

column_info = get_column_info(train_df_processed)
print("column_info")
print(column_info)


column_info
{'Category': ['Name', 'Ticket', 'Cabin'], 'Numeric': ['PassengerId', 'Survived', 'Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], 'Datetime': [], 'Others': []}


In [5]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

# Assuming train_df_processed and eval_df_processed are already defined from previous tasks
X_train = train_df_processed.drop(columns=['Survived', 'PassengerId'])
y_train = train_df_processed['Survived']

X_eval = eval_df_processed.drop(columns=['Survived', 'PassengerId'])
y_eval = eval_df_processed['Survived']

# Initialize and train the XGBoost model
model = XGBClassifier(
    n_estimators=100,
    max_depth=6,
    learning_rate=0.1,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42
)

model.fit(X_train, y_train)

# Predict on the eval data
y_pred = model.predict(X_eval)

# Calculate accuracy
accuracy = accuracy_score(y_eval, y_pred)
print(f"Accuracy on eval data: {accuracy:.4f}")


ValueError: DataFrame.dtypes for data must be int, float, bool or category. When categorical type is supplied, the experimental DMatrix parameter`enable_categorical` must be set to `True`.  Invalid columns:Name: object, Ticket: object, Cabin: object